# Getting Started with ySights

This tutorial introduces the basics of ySights, a Python library for analyzing data from YSocial simulations.

## What You'll Learn

- How to initialize `YDataHandler`
- Loading and exploring simulation data
- Working with agents and posts
- Using opaque identifiers from the active dataset
- Built-in summaries, cache diagnostics, and index suggestions

## Prerequisites

You need:
- ySights installed (`pip install ysights`)
- A YSocial simulation database file (`.db` format)

---

## 1. Importing ySights

First, let's import the main components we'll be using:

In [ ]:
from pathlib import Path

from ysights import YDataHandler
import matplotlib.pyplot as plt
import numpy as np

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 2. Initializing the Data Handler

The `YDataHandler` is your main interface to the simulation database.

**Note**: The examples below use the bundled tutorial database when available.

In [ ]:
# Initialize the data handler
from pathlib import Path


def resolve_example_db():
    candidates = [
        Path("ysocial_db.db"),
        Path("../notebooks/ysocial_db.db"),
        Path("../../notebooks/ysocial_db.db"),
        Path("docs/notebooks/ysocial_db.db"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate.resolve())
    return "ysocial_db.db"

db_path = resolve_example_db()

try:
    ydh = YDataHandler(db_path)
    print("✓ Successfully connected to the database!")
except FileNotFoundError:
    print("✗ Database file not found. Please check the path.")
    print("  For this tutorial, we'll show the expected outputs.")

## 3. Exploring the Simulation

Let's get some basic information about the simulation.

In [ ]:
time_range = ydh.time_range()
print("Simulation Time Range:")
print(f"  Min Round: {time_range['min_round']}")
print(f"  Max Round: {time_range['max_round']}")
print(f"  Duration: {time_range['max_round'] - time_range['min_round']} rounds")

In [ ]:
num_agents = ydh.number_of_agents()
print(f"Total Agents in Simulation: {num_agents}")

## 4. Working with Agents

Agents represent the users in the simulation. Let's explore their properties.

In [ ]:
agents = ydh.agents()
print(f"Retrieved {len(agents.get_agents())} agents")

first_agent = agents.get_agents()[0]
print("\nFirst Agent Properties:")
print(f"  ID: {first_agent.id}")
print(f"  Age: {first_agent.age}")
print(f"  Gender: {first_agent.gender}")
print(f"  Education: {first_agent.education}")

### Filtering Agents by Feature

In [ ]:
young_agents = ydh.agents_by_feature('age', 25)
print(f"Agents aged 25: {len(young_agents.get_agents())}")

female_agents = ydh.agents_by_feature('gender', 'F')
print(f"Female agents: {len(female_agents.get_agents())}")

### Age Distribution Visualization

In [ ]:
ages = [agent.age for agent in agents.get_agents()]

plt.figure(figsize=(10, 6))
plt.hist(ages, bins=20, edgecolor='black', alpha=0.7)
plt.xlabel('Age', fontsize=12)
plt.ylabel('Number of Agents', fontsize=12)
plt.title('Age Distribution of Agents', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

## 5. Working with Posts

Posts represent the content created by agents in the simulation.

In [ ]:
agent_id = next(iter(ydh.agent_mapping()))
agent_posts = ydh.posts_by_agent(agent_id)

print(f"Agent {agent_id} created {len(agent_posts.get_posts())} posts")

if agent_posts.get_posts():
    first_post = agent_posts.get_posts()[0]
    print("\nFirst Post Details:")
    print(f"  Post ID: {first_post.id}")
    print(f"  Author: {first_post.user_id}")
    print(f"  Round: {first_post.round}")
    print(f"  Topic: {first_post.topics}")
    print(f"  Emotion: {first_post.emotions}")

## 6. Agent Interest Profiles

Each agent has an interest profile showing their engagement with different topics.

In [ ]:
profile = ydh.agent_interests(agent_id)

print(f"Interest Profile for Agent {agent_id}:")
for topic, score in list(profile.items())[:5]:
    print(f"  Topic {topic}: {score:.3f}")

### Visualizing Interest Profile

In [ ]:
sorted_topics = sorted(profile.items(), key=lambda x: x[1], reverse=True)[:10]
topics = [f"Topic {t[0]}" for t in sorted_topics]
scores = [t[1] for t in sorted_topics]

plt.figure(figsize=(12, 6))
plt.barh(topics, scores, color='steelblue', alpha=0.8)
plt.xlabel('Interest Score', fontsize=12)
plt.ylabel('Topics', fontsize=12)
plt.title(f'Top 10 Topics for Agent {agent_id}', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 7. Custom Queries

For more complex analysis, you can execute custom SQL queries:

In [ ]:
query = """
    SELECT user_id, COUNT(*) as post_count 
    FROM post 
    GROUP BY user_id 
    ORDER BY post_count DESC 
    LIMIT 5
"""

results = ydh.custom_query(query)
print("Top 5 Most Active Agents:")
for i, row in enumerate(results, 1):
    print(f"  {i}. Agent {row[0]}: {row[1]} posts")

## 8. Built-in Summaries and Diagnostics

The current ySights API includes dataset-level diagnostics that are useful before deeper analysis.

In [ ]:
summary_report = ydh.summary_report()
summary_frame = ydh.summary_frame()
cache_info = ydh.analysis_cache_info()
recommended_indexes = ydh.recommended_indexes()
benchmark = ydh.benchmark_analytics(iterations=1)

print("Summary Report (selected keys):")
for key in ["agent_count", "post_count", "thread_count", "report_count", "forum_session_count"]:
    if key in summary_report:
        print(f"  {key}: {summary_report[key]}")

print("\nSummary Frame Preview:")
print(summary_frame.head().to_string(index=False))

print("\nCache Diagnostics:")
print(cache_info)

print("\nRecommended Indexes:")
print(recommended_indexes)

print("\nBenchmark Metrics:")
print(list(benchmark["metrics"].keys()))

## Summary

In this tutorial, you learned:

✓ How to initialize `YDataHandler` with your simulation database  
✓ Basic exploration of simulation time range and agent count  
✓ Working with `Agents` and filtering by features  
✓ Retrieving and examining `Posts`  
✓ Using opaque identifiers that work across numeric and UUID-like databases  
✓ Analyzing agent interest profiles  
✓ Creating visualizations of simulation data  
✓ Executing custom SQL queries for advanced analysis  
✓ Inspecting built-in summaries, cache diagnostics, and recommended indexes  

## Next Steps

Continue with:
- **Network Analysis Tutorial**: Learn how to extract and analyze social networks
- **Algorithms Tutorial**: Explore profile similarity, topic lifecycle, and moderation metrics
- **Visualization Tutorial**: Create advanced visualizations of simulation data